# 📦 Setup & Libraries
All imports required for the scraping and matching pipeline.


In [12]:
from rapidfuzz import fuzz, process
from selenium.webdriver.common.by import By
import os
import pandas as pd
import re
import time
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By


# 🕵🏻‍♀️ Phase 1: Web Scraping the Full Catalog
Scrapes all 55 pages of the e-commerce catalog, extracting product names, prices, brands, and URLs. Bypasses bot protections using `undetected_chromedriver`.


In [19]:

options = uc.ChromeOptions()
options.add_argument("--window-size=1920,1080")
driver = uc.Chrome(options=options)

TOTAL_PAGES = 55  # Set to 2 or 3 first to test, then run all 55
base_url = "https://korean-skincare.de/collections/all?page="

scraped_records = []

try:
    for page in range(1, TOTAL_PAGES + 1):
        target_url = f"{base_url}{page}"
        print(f"Scraping Page {page}/{TOTAL_PAGES}: {target_url}")
        
        driver.get(target_url)
        time.sleep(6)  # Give JS grid time to render
        
        product_cards = driver.find_elements(By.CSS_SELECTOR, ".product-item")
        print(f"  Found {len(product_cards)} card elements on page {page}")

        for card in product_cards:
            # Title and Link
            title_el = card.find_elements(By.CSS_SELECTOR, "a.card-title")
            if not title_el:
                continue
            title = title_el[0].text.strip()
            if not title:
                continue
            link = title_el[0].get_attribute("href")

            # Brand / Vendor
            vendor_el = card.find_elements(By.CSS_SELECTOR, ".card-vendor a")
            brand = vendor_el[0].text.strip() if vendor_el else None

            # Prices
            sale_container = card.find_elements(By.CSS_SELECTOR, ".price__sale")
            reg_container = card.find_elements(By.CSS_SELECTOR, ".price__regular")

            current_price = None
            orig_price = None

            if sale_container:
                compare_tag = card.find_elements(By.CSS_SELECTOR, "dd[data-compare]")
                last_tag = card.find_elements(By.CSS_SELECTOR, "dd[data-last]")

                if compare_tag:
                    val = compare_tag[0].get_attribute("data-compare") or compare_tag[0].get_dom_attribute("data-compare")
                    if val:
                        try:
                            orig_price = float(val.strip()) / 100.0
                        except (ValueError, TypeError):
                            pass

                if last_tag:
                    val = last_tag[0].get_attribute("data-last") or last_tag[0].get_dom_attribute("data-last")
                    if val:
                        try:
                            current_price = float(val.strip()) / 100.0
                        except (ValueError, TypeError):
                            pass

                if current_price is None and last_tag and last_tag[0].text.strip():
                    clean_txt = last_tag[0].text.replace("€", "").replace(",", ".").strip()
                    try:
                        current_price = float(clean_txt)
                    except (ValueError, TypeError):
                        pass

            elif reg_container:
                price_tag = reg_container[0].find_elements(By.CSS_SELECTOR, ".price-item")
                if price_tag and price_tag[0].text.strip():
                    clean_txt = price_tag[0].text.replace("€", "").replace(",", ".").strip()
                    try:
                        current_price = float(clean_txt)
                        orig_price = current_price
                    except (ValueError, TypeError):
                        pass

            if orig_price is None:
                orig_price = current_price
            if current_price is None:
                current_price = orig_price

            scraped_records.append({
                "Brand": brand,
                "Product Name": title,
                "Current Price": current_price,
                "Original Price": orig_price,
                "URL": link
            })

    # Assemble and clean DataFrame
    df = pd.DataFrame(scraped_records)
    
    # Deduplicate desktop vs mobile cards
    df = df.drop_duplicates(subset=["URL"]).reset_index(drop=True)

    # Calculate discount
    df["Discount (%)"] = (
        (df["Original Price"] - df["Current Price"])
        / df["Original Price"]
        * 100
    ).round(1)

    print(f"\nDone! Scraped {len(df)} unique products.")
    df.to_csv("korean_skincare_catalog.csv", index=False)
    print("Saved catalog to 'korean_skincare_catalog.csv'.")

finally:
    driver.quit()

Scraping Page 1/55: https://korean-skincare.de/collections/all?page=1
  Found 82 card elements on page 1
Scraping Page 2/55: https://korean-skincare.de/collections/all?page=2
  Found 82 card elements on page 2
Scraping Page 3/55: https://korean-skincare.de/collections/all?page=3
  Found 82 card elements on page 3
Scraping Page 4/55: https://korean-skincare.de/collections/all?page=4
  Found 82 card elements on page 4
Scraping Page 5/55: https://korean-skincare.de/collections/all?page=5
  Found 82 card elements on page 5
Scraping Page 6/55: https://korean-skincare.de/collections/all?page=6
  Found 82 card elements on page 6
Scraping Page 7/55: https://korean-skincare.de/collections/all?page=7
  Found 82 card elements on page 7
Scraping Page 8/55: https://korean-skincare.de/collections/all?page=8
  Found 82 card elements on page 8
Scraping Page 9/55: https://korean-skincare.de/collections/all?page=9
  Found 82 card elements on page 9
Scraping Page 10/55: https://korean-skincare.de/collect

# 🧬 Phase 2: Data Enrichment via Database Matching
Merges scraped storefront items with a 19k+ row INCI database using Exact Slug matching and RapidFuzz token sorting to retrieve base ingredients and skin concerns.


In [ ]:

print("Loading datasets...")
df_shop = pd.read_csv("korean_skincare_catalog.csv")
df_db = pd.read_csv("datasheet.csv")

# 1. Generate clean slugs for exact matching
def generate_slug(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return text

# Extract slug from Shopify URL: .../products/dr-althea-345-relief-cream -> dr-althea-345-relief-cream
df_shop["shop_slug"] = (
    df_shop["URL"].astype(str).str.rstrip("/").str.split("/").str[-1]
)

# Build slug in DB from Brand + Name
df_db["db_slug"] = [
    generate_slug(f"{b} {n}") for b, n in zip(df_db["brand"], df_db["name"])
]

# Drop duplicate slugs in df_db before creating dictionary lookup
df_db_unique_slugs = df_db.drop_duplicates(subset=["db_slug"]).copy()
slug_lookup = df_db_unique_slugs.set_index("db_slug").to_dict(orient="index")

# 2. Text normalization for fuzzy fallback
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())

df_shop["clean_brand"] = df_shop["Brand"].apply(clean_text)
df_shop["clean_name"] = df_shop["Product Name"].apply(clean_text)

df_db["clean_brand"] = df_db["brand"].apply(clean_text)
df_db["clean_name"] = df_db["name"].apply(clean_text)

# Pre-group database products by brand for fast candidate filtering
db_by_brand = {}
for brand, group in df_db.groupby("clean_brand"):
    db_by_brand[brand] = dict(zip(group.index, group["clean_name"]))

print("Merging scraped items with database...")
enriched_records = []
exact_matches = 0
fuzzy_matches = 0
unmatched = 0

for _, shop_row in df_shop.iterrows():
    slug = shop_row["shop_slug"]
    brand = shop_row["clean_brand"]
    name = shop_row["clean_name"]

    matched_row = None

    # Tier 1: Exact slug match
    if slug in slug_lookup:
        matched_row = slug_lookup[slug]
        exact_matches += 1
    else:
        # Tier 2: Fuzzy match within brand candidates
        candidates = db_by_brand.get(brand, {})
        if candidates:
            best_match = process.extractOne(
                name, candidates, scorer=fuzz.token_sort_ratio
            )
            if best_match and best_match[1] >= 72:  # Similarity threshold
                matched_idx = best_match[2]
                matched_row = df_db.loc[matched_idx].to_dict()
                fuzzy_matches += 1

    if matched_row is None:
        unmatched += 1

    enriched_records.append({
        "Brand": shop_row["Brand"],
        "Product Name": shop_row["Product Name"],
        "Current Price": shop_row["Current Price"],
        "Original Price": shop_row["Original Price"],
        "Discount (%)": shop_row["Discount (%)"],
        "URL": shop_row["URL"],
        "Category": matched_row["type"] if matched_row else None,
        "Skin Concerns": matched_row["afterUse"] if matched_row else None,
        "Ingredients": matched_row["ingridients"] if matched_row else None,
    })

df_enriched = pd.DataFrame(enriched_records)
df_enriched.to_csv("korean_skincare_enriched.csv", index=False)

print("\n--- Merge Summary ---")
print(f"Total Scraped Products: {len(df_shop)}")
print(f"Exact Matches:          {exact_matches}")
print(f"Fuzzy Matches:          {fuzzy_matches}")
print(f"Unmatched:              {unmatched}")
print(f"Match Rate:             {((exact_matches + fuzzy_matches) / len(df_shop)) * 100:.1f}%")
print("Saved enriched data to 'korean_skincare_enriched.csv'.")

# 🔍 Phase 3: Targeted Deep Scraping for Missing Ingredients
For products that didn't match the database, this script goes directly to the UK site (`koreanskincare.co.uk`) to retrieve missing English INCI formulas.


In [34]:

INPUT_FILE = "korean_skincare_enriched.csv"
OUTPUT_FILE = "korean_skincare_enriched_completed.csv"
CHECKPOINT_EVERY = 25

# 1. Resume from existing output if present, otherwise load input file
if os.path.exists(OUTPUT_FILE):
    print(f"Resuming from existing '{OUTPUT_FILE}'...")
    df = pd.read_csv(OUTPUT_FILE)
else:
    print(f"Starting fresh from '{INPUT_FILE}'...")
    df = pd.read_csv(INPUT_FILE)

# Identify rows that still need any of the 3 fields
missing_mask = (
    df["Category"].isna()
    | df["Skin Concerns"].isna()
    | df["Ingredients"].isna()
)
indices_to_scrape = df[missing_mask].index.tolist()

print(f"Total products in catalog: {len(df)}")
print(f"Remaining products to scrape: {len(indices_to_scrape)}")

if not indices_to_scrape:
    print("All products are already fully enriched!")
    exit()

# 2. Chrome Setup (headless for speed)
options = uc.ChromeOptions()
options.add_argument("--window-size=1920,1080")
options.add_argument("--headless=new")
driver = uc.Chrome(options=options)

try:
    for count, idx in enumerate(indices_to_scrape, start=1):
        de_url = str(df.at[idx, "URL"])
        en_url = de_url.replace("korean-skincare.de", "koreanskincare.co.uk")

        print(f"[{count}/{len(indices_to_scrape)}] Processing index {idx}: {en_url}")

        try:
            driver.get(en_url)
            time.sleep(2.5)

            # --- A. Product Type ---
            product_type = None
            info_items = driver.find_elements(By.CSS_SELECTOR, ".productView-info-item")
            for item in info_items:
                txt = item.get_attribute("textContent")
                if "Product Type" in txt:
                    val_el = item.find_elements(By.CSS_SELECTOR, ".productView-info-value")
                    if val_el:
                        product_type = val_el[0].get_attribute("textContent").strip()
                        break

            # --- B. Paragraphs for Concerns & Ingredients ---
            paragraphs = driver.find_elements(
                By.CSS_SELECTOR,
                "#tab-description p, #tab-description-mobile p, .productView-desc p, [id*='description'] p"
            )

            concerns = None
            ingredients = None

            for p in paragraphs:
                raw_text = p.get_attribute("textContent").strip()
                if not raw_text:
                    continue

                # Recommended for / Concerns
                if "Recommended for" in raw_text or "Empfohlen für" in raw_text:
                    cleaned = raw_text.split(":", 1)[-1].strip().strip('"').strip("'")
                    if cleaned:
                        concerns = cleaned

                # Ingredients
                if "Ingredients:" in raw_text:
                    cleaned_inci = raw_text.split("Ingredients:", 1)[-1].strip().strip('"').strip("'")
                    if len(cleaned_inci) > 20:
                        ingredients = cleaned_inci
                elif any(marker in raw_text for marker in ["Aqua", "Water", "Centella Asiatica"]):
                    if ":" in raw_text and len(raw_text.split(":", 1)[0]) <= 12:
                        cleaned_inci = raw_text.split(":", 1)[1].strip()
                    else:
                        cleaned_inci = raw_text
                    cleaned_inci = cleaned_inci.strip().strip('"').strip("'")
                    if len(cleaned_inci) > 20 and not ingredients:
                        ingredients = cleaned_inci

            # --- C. Update DataFrame ---
            if product_type and pd.isna(df.at[idx, "Category"]):
                df.at[idx, "Category"] = product_type

            if concerns and pd.isna(df.at[idx, "Skin Concerns"]):
                df.at[idx, "Skin Concerns"] = concerns

            if ingredients and pd.isna(df.at[idx, "Ingredients"]):
                df.at[idx, "Ingredients"] = ingredients

        except Exception as e:
            print(f"  --> Warning at index {idx}: {e}")

        # Checkpoint saving
        if count % CHECKPOINT_EVERY == 0:
            df.to_csv(OUTPUT_FILE, index=False)
            print(f"  [Checkpoint] Saved {count} processed rows to '{OUTPUT_FILE}'.")

finally:
    driver.quit()
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nScraping complete. Final dataset saved to '{OUTPUT_FILE}'.")

Starting fresh from 'korean_skincare_enriched.csv'...
Total products in catalog: 2713
Remaining products to scrape: 1675
[1/1675] Processing index 0: https://koreanskincare.co.uk/products/korean-skincare-the-tidal-glow-advent-calendar
[2/1675] Processing index 1: https://koreanskincare.co.uk/products/beauty-of-joseon-relief-sun-rice-niacinamide
[3/1675] Processing index 2: https://koreanskincare.co.uk/products/dr-althea-345-relief-cream
[4/1675] Processing index 4: https://koreanskincare.co.uk/products/anua-heartleaf-quercetinol-pore-deep-cleansing-foam
[5/1675] Processing index 11: https://koreanskincare.co.uk/products/anua-niacinamide-10-txa-4-serum
[6/1675] Processing index 12: https://koreanskincare.co.uk/products/missha-airy-fit-sheet-mask
[7/1675] Processing index 13: https://koreanskincare.co.uk/products/beauty-of-joseon-relief-sun-aqua-fresh-rice-b5
[8/1675] Processing index 14: https://koreanskincare.co.uk/products/anua-azelaic-acid-10-hyaluron-redness-soothing-serum
[9/1675] 

# ⭐ Phase 4: Scraping Judge.me Customer Ratings
Extracts verified customer star ratings and review counts from the storefront's Judge.me widgets.


In [63]:

reviews_data = []
TOTAL_PAGES = 55  # adjust if there are fewer/more collection pages

for page_num in range(1, TOTAL_PAGES + 1):
    page_url = f"https://korean-skincare.de/collections/all?page={page_num}"
    driver.get(page_url)
    time.sleep(2.5)  # give Judge.me time to inject attributes

    cards = driver.find_elements(By.CSS_SELECTOR, ".card.ajax-loaded")
    if not cards:
        # Fallback locator
        all_items = driver.find_elements(By.CSS_SELECTOR, ".product-item, .card")
        cards = [c for c in all_items if len(c.find_elements(By.CSS_SELECTOR, "a[href*='/products/']")) > 0]

    for card in cards:
        link_els = card.find_elements(By.CSS_SELECTOR, "a[href*='/products/']")
        if not link_els:
            continue
        
        href = link_els[0].get_attribute("href")
        if not href or "/products/" not in href:
            continue
            
        # Standardize URL (strip query parameters like ?variant=...)
        clean_url = href.split("?")[0].strip()

        rating = None
        reviews = None

        badge = card.find_elements(By.CSS_SELECTOR, ".jdgm-prev-badge")
        if badge:
            raw_rating = badge[0].get_attribute("data-average-rating")
            raw_reviews = badge[0].get_attribute("data-number-of-reviews")

            if raw_rating:
                try:
                    rating = float(raw_rating.strip())
                except ValueError:
                    pass

            if raw_reviews:
                try:
                    reviews = int(raw_reviews.strip())
                except ValueError:
                    pass

        reviews_data.append({
            "URL": clean_url,
            "Rating": rating,
            "Review_Count": reviews
        })

    print(f"Page {page_num}/{TOTAL_PAGES} processed ({len(cards)} items found).")

# Deduplicate and save
df_reviews = pd.DataFrame(reviews_data).drop_duplicates(subset=["URL"])
df_reviews.to_csv("product_reviews.csv", index=False)
print(f"\nSaved {len(df_reviews)} product reviews to 'product_reviews.csv'.")

Page 1/55 processed (140 items found).
Page 2/55 processed (140 items found).
Page 3/55 processed (140 items found).
Page 4/55 processed (140 items found).
Page 5/55 processed (140 items found).
Page 6/55 processed (140 items found).
Page 7/55 processed (140 items found).
Page 8/55 processed (140 items found).
Page 9/55 processed (140 items found).
Page 10/55 processed (140 items found).
Page 11/55 processed (140 items found).
Page 12/55 processed (140 items found).
Page 13/55 processed (140 items found).
Page 14/55 processed (140 items found).
Page 15/55 processed (140 items found).
Page 16/55 processed (140 items found).
Page 17/55 processed (140 items found).
Page 18/55 processed (140 items found).
Page 19/55 processed (140 items found).
Page 20/55 processed (140 items found).
Page 21/55 processed (140 items found).
Page 22/55 processed (140 items found).
Page 23/55 processed (140 items found).
Page 24/55 processed (140 items found).
Page 25/55 processed (140 items found).
Page 26/5

# 🔗 Phase 5: Final Raw Dataset Merge
Merges the enriched product catalog with the scraped customer reviews. Outputs the final raw dataset (`korean_skincare_raw_scraped.csv`) ready for the Data Cleaning notebook.


In [3]:
# 1. Load the main enriched dataset
if os.path.exists("korean_skincare_enriched_completed.csv"):
    main_file = "korean_skincare_enriched_completed.csv"
else:
    main_file = "korean_skincare_enriched.csv"

df_main = pd.read_csv(main_file)
df_reviews = pd.read_csv("product_reviews.csv")

print(f"Loaded main dataset ('{main_file}'): {len(df_main)} rows")
print(f"Loaded reviews dataset: {len(df_reviews)} rows")

# 2. Normalize URLs on both datasets (strip query params, trailing slashes, whitespace)
def normalize_url(url_series):
    return (
        url_series.astype(str)
        .str.split("?").str[0]
        .str.rstrip("/")
        .str.strip()
    )

df_main["url_clean"] = normalize_url(df_main["URL"])
df_reviews["url_clean"] = normalize_url(df_reviews["URL"])

# Deduplicate reviews on the normalized key just in case
df_reviews_unique = df_reviews.drop_duplicates(subset=["url_clean"]).copy()

# 3. Merge rating and review counts onto the main DataFrame
df_merged = df_main.merge(
    df_reviews_unique[["url_clean", "Rating", "Review_Count"]],
    on="url_clean",
    how="left"
).drop(columns=["url_clean"])

# 4. Check match rate and save
matched_count = df_merged["Rating"].notna().sum()
print(f"Successfully matched review data for {matched_count} / {len(df_merged)} products ({matched_count / len(df_merged) * 100:.1f}%).")

output_filename = "korean_skincare_raw_scraped.csv"
df_merged.to_csv(output_filename, index=False)
print(f"Master file saved to '{output_filename}'.")

Loaded main dataset ('korean_skincare_enriched_completed.csv'): 2713 rows
Loaded reviews dataset: 2611 rows
Successfully matched review data for 2611 / 2713 products (96.2%).
Master file saved to 'korean_skincare_raw_scraped.csv'.


# 🩹 Phase 6: Targeted Scraping for Remaining Missing Ratings
Directly opens product pages for the remaining items with missing ratings to inspect and extract the individual Judge.me rating badges.

In [14]:
# 1. Load the dataset and find missing URLs
df = pd.read_csv("korean_skincare_raw_scraped.csv")
nan_mask = df["Rating"].isna()
missing_urls = df.loc[nan_mask, "URL"].dropna().unique().tolist()
total_missing = len(missing_urls)

print(f"Products with missing ratings to scrape: {total_missing}")

# 2. Initialize the browser driver
options = uc.ChromeOptions()
options.add_argument("--window-size=1920,1080")
driver = uc.Chrome(options=options, version_main=156)

for i, url in enumerate(missing_urls, 1):
    rating = None
    reviews = None
    
    try:
        driver.get(url)
        time.sleep(2.0)  # Allow Judge.me scripts to inject DOM attributes

        # Grab all badge elements
        badges = driver.find_elements(By.CSS_SELECTOR, ".jdgm-prev-badge, .jdgm-widget")

        # Iterate through badges to find the populated one
        for b in badges:
            raw_rating = b.get_attribute("data-average-rating")
            raw_reviews = b.get_attribute("data-number-of-reviews")

            if raw_rating and raw_rating.strip() not in ["", "None"]:
                try:
                    rating = float(raw_rating.strip())
                except ValueError:
                    pass

            if raw_reviews and raw_reviews.strip() not in ["", "None"]:
                try:
                    reviews = int(raw_reviews.strip())
                except ValueError:
                    pass

            # Once we found valid data, break out of the badge loop
            if rating is not None and reviews is not None:
                break

        # Fallback: Check if the product explicitly has no reviews yet
        if rating is None:
            no_reviews_text = driver.find_elements(By.CSS_SELECTOR, ".jdgm-prev-badge__text")
            if no_reviews_text and "no review" in no_reviews_text[0].text.lower():
                rating = 0.0
                reviews = 0

        scraped_data[url] = {"Rating": rating, "Review_Count": reviews}
        
        slug = url.split("/products/")[-1][:30]
        print(f"[{i}/{total_missing}] {slug:<30} -> Rating: {rating} ★ | Reviews: {reviews}")

    except Exception as e:
        print(f"[{i}/{total_missing}] Error scraping {url}: {e}")

# 1. Update df_merged with the new values
for url, stats in scraped_data.items():
    if stats["Rating"] is not None:
        df.loc[df["URL"] == url, "Rating"] = stats["Rating"]
        df.loc[df["URL"] == url, "Review_Count"] = stats["Review_Count"]

print(f"\nFinished! Remaining NaN ratings in df_merged: {df['Rating'].isna().sum()}")

Products with missing ratings to scrape: 102
[1/102] purito-seoul-multi-pdrn-collag -> Rating: 4.33 ★ | Reviews: 3
[2/102] illiyoon-ceramide-ato-soothing -> Rating: 5.0 ★ | Reviews: 2
[3/102] vt-cosmetics-pdrn-reedle-shot- -> Rating: 2.0 ★ | Reviews: 1
[4/102] kundal-progen-double-bonding-h -> Rating: 4.0 ★ | Reviews: 3
[5/102] lagom-airy-velvet-sun-stick    -> Rating: 0.0 ★ | Reviews: 0
[6/102] dr-melaxin-astaxanthin-brighte -> Rating: 4.33 ★ | Reviews: 3
[7/102] heveblue-salmon-caring-centell -> Rating: 5.0 ★ | Reviews: 3
[8/102] farm-stay-collagen-water-full- -> Rating: 5.0 ★ | Reviews: 3
[9/102] dr-ceuracle-pure-vc-mellight-b -> Rating: 4.88 ★ | Reviews: 8
[10/102] dr-ceuracle-hyal-reyouth-toner -> Rating: 4.62 ★ | Reviews: 13
[11/102] barrierderm-mild-foam-cleanser -> Rating: 3.6 ★ | Reviews: 5
[12/102] neogen-dermalogy-probiotics-yo -> Rating: 4.17 ★ | Reviews: 12
[13/102] dr-reju-all-advanced-pdrn-reju -> Rating: 0.0 ★ | Reviews: 0
[14/102] dr-althea-pure-retinol-0-15-cr -> Rati

# 🧪 Phase 7: Targeted Scraping for Remaining Missing Ingredients
Visits the UK store pages for items still missing ingredients, clicks open the 'Ingredients' / 'Inhaltsstoffe' accordions via JavaScript, and grabs the complete chemical lists.


In [18]:
# Filter rows where Ingredients is NaN, empty, or whitespace
nan_ing_mask = df["Ingredients"].isna() | (df["Ingredients"].astype(str).str.strip() == "")
missing_ing_urls = df.loc[nan_ing_mask, "URL"].dropna().unique().tolist()
print(f"Products missing ingredients to scrape: {len(missing_ing_urls)}")


Products missing ingredients to scrape: 171


In [19]:
# Scrape via UK domain by expanding ingredient accordions
for i, de_url in enumerate(missing_ing_urls, 1):
    uk_url = de_url.replace("korean-skincare.de", "korean-skincare.co.uk")
    try:
        driver.get(uk_url)
        time.sleep(2)

        # Open Ingredients tab if collapsed
        driver.execute_script("""
            document.querySelectorAll('summary, button, .accordion__title').forEach(el => {
                if (el.innerText.toLowerCase().includes('ingredient')) el.click();
            });
        """)
        time.sleep(0.5)

        elements = driver.find_elements(By.CSS_SELECTOR, ".accordion__content, .product__description, div")
        for el in elements:
            text = el.text.strip()
            if any(k in text.lower() for k in ["water", "aqua", "glycerin", "centella", "niacinamide"]):
                clean_text = text.split(":")[-1].strip() if "ingredient" in text.lower() else text
                df.loc[df["URL"] == de_url, "Ingredients"] = clean_text
                print(f"[{i}/{len(missing_ing_urls)}] Found: {de_url.split('/products/')[-1][:30]}")
                break
    except Exception as e:
        print(f"[{i}/{len(missing_ing_urls)}] Error on {uk_url}: {e}")

print(f"Done! Remaining NaN ingredients: {df['Ingredients'].isna().sum()}")

[3/171] Found: some-by-mi-clear-spot-patch
[4/171] Found: purito-seoul-wonder-releaf-cen
[5/171] Found: kundal-macadamia-ultra-hair-se
[6/171] Found: pyunkang-yul-acne-spot-patch-s
[7/171] Found: kundal-shampoo-brush
[8/171] Found: innisfree-no-sebum-mineral-pow
[10/171] Found: benton-hanji-body-wash-towel
[16/171] Found: tocobo-glass-tinted-lip-balm
[19/171] Found: tocobo-powder-cream-lip-balm
[23/171] Found: real-barrier-extreme-cream-ori
[24/171] Found: tfit-cover-up-pro-concealer
[25/171] Found: some-by-mi-diamond-brightening
[26/171] Found: ilso-dual-clean-brush
[29/171] Found: farm-stay-real-shea-butter-ess
[30/171] Found: kaine-glow-melting-lip-balm
[31/171] Found: black-rouge-drop-eyes-stick-gl
[33/171] Found: needly-tint-lip-balm
[37/171] Found: etude-curl-fix-mascara
[39/171] Found: black-rouge-chewy-lip-glaze
[40/171] Found: kundal-volume-fixing-curl-crea
[42/171] Found: needly-mild-multi-face-towel
[45/171] Found: abib-airy-sunstick-smoothing-b
[46/171] Found: unleashia-moo

In [20]:
# 3. Save
df.to_csv("korean_skincare_raw_scraped_old.csv", index=False)
print(f"Done! Remaining NaN ingredients: {df['Ingredients'].isna().sum()}")

Done! Remaining NaN ingredients: 112
